In [0]:
from pyspark.sql.functions import current_timestamp
def add_ingestion_date(input_df):
    output_df = input_df.withColumn("ingestion_date", current_timestamp())
    return output_df

In [0]:
def delete_records_by_filter(schema_name,table_name, filter_column, filter_value):
    if spark.catalog.tableExists(f"{schema_name}.{table_name}"):
        spark.sql(f"DELETE FROM {schema_name}.{table_name} WHERE {filter_column} = '{filter_value}'")

In [0]:
def merge_delta_lake(schema_name,table_name,src_dataframe, merge_condition, partition_column):
    from delta.tables import DeltaTable
    #Si existe la tabla aplicamos un merge.
    if spark.catalog.tableExists(f"{schema_name}.{table_name}"): 
        tgt = DeltaTable.forName(spark, f"{schema_name}.{table_name}")   
        tgt.alias('tgt') \
        .merge(
            src_dataframe.alias('src'),merge_condition
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
    else:
        src_dataframe.write.mode("overwrite").partitionBy(partition_column).format("delta").saveAsTable(f"{schema_name}.{table_name}")